# Physics super-resolution with VAR (next-scale prediction), model v2

**What changed from v1.** The v1 output had a faint background over the whole image, no tracks, and track energy showing up in ECAL. v2 changes the tokenizer:

* the decoder predicts **hit / no hit** for every cell, plus the energy if hit. Empty cells are exactly zero, like a detector after zero suppression;
* the energy loss is **per channel**, so energy cannot move from tracks to ECAL;
* the latent is **32×32** (one token per 4×4 pixels instead of 8×8), and the decoder also sees the **LR image**;
* the main SR output uses the **most likely token** (greedy); sampled tokens are a test-only diagnostic.

**Where results go.** Every training campaign is one **experiment** with its own dated folder on Drive:

```
MyDrive/GSoC_SuperRes/
  experiments/<dataset>/<version>/<date>-<tag>/     e.g. experiments/qg/v2/2026-10-05-tokfix/
      README_layout.md   run_info.json   data_meta.json
      runs/   optuna/   summary/
  smoke_tests/...                                   same layout, for SMOKE / FAKE_DATA runs
```

Every run, figure and table name starts with `<dataset>__<version>__<date>-<tag>`, e.g. `qg__v2__2026-10-05-tokfix__vqvae__hr__s42`, so a figure copied out of Drive still says which experiment it belongs to. `run_info.json` records when the experiment started and, for every session, the code commit, GPU and full config. The v1 results in `MyDrive/superres_results/` are not touched. Old checkpoints can't be reused: v2 trains the tokenizer and VAR from scratch.

Datasets: **CMS quark/gluon jet images** (`qg`, 3 x 125 x 125) and **CaloChallenge Dataset 2** (`calo`, 45 x 16 x 9 showers).

Pipeline, one cell per stage:

`download -> prepare -> tune_vqvae -> train_vqvae -> tune_var -> train_var -> generate -> eval_sr -> tune_tagger -> train_taggers -> summarize`

* Use a **GPU runtime** (Runtime -> Change runtime type -> T4 / L4 / A100).
* First run with `SMOKE = True` (tiny subset, 1-2 epochs) to check everything works, then set `SMOKE = False` and run all cells again.
* All results (checkpoints, metrics, Optuna studies, figures) go to Google Drive. Training and tuning resume from Drive after a disconnect, so after a disconnect just re-run the cells.
* Run one dataset per session: change `DATASET` and restart the runtime to switch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU - switch the runtime to GPU'
!df -h /content | tail -1

from google.colab import drive
drive.mount('/content/drive')

## 1. Get the code from GitHub

The first run clones the repository into `/content/superres_code`. Every re-run pulls the latest commit of `BRANCH`, so code pushed to GitHub is picked up without restarting the runtime.

In [ ]:
GIT_URL = 'https://github.com/rajveer43/physics-superres-var.git'
BRANCH  = 'v2-eval-tagger-checks'
CODE    = '/content/superres_code'

import os, shutil, subprocess, sys

def git(*args, cwd=CODE):
    r = subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr}")
    return r.stdout.strip()

if os.path.isdir(os.path.join(CODE, '.git')):
    git('fetch', '--depth', '1', 'origin', BRANCH)
    git('reset', '--hard', f'origin/{BRANCH}')   # pull; discards any edits made inside Colab
else:
    shutil.rmtree(CODE, ignore_errors=True)
    git('clone', '--depth', '1', '--branch', BRANCH, GIT_URL, CODE, cwd='/content')
print('code at commit:', git('log', '-1', '--format=%h %s (%cr)'))

if CODE not in sys.path:
    sys.path.insert(0, CODE)
for m in [m for m in sys.modules if m.split('.')[0] in ('superres', 'tests')]:
    del sys.modules[m]   # re-import the freshly pulled code

In [ ]:
# Colab already has torch, numpy, scipy, sklearn, pandas, matplotlib, h5py, pyarrow, requests
!pip install -q optuna tabulate
import torch, optuna
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| optuna', optuna.__version__)

## 2. Settings

Every value in `superres/config.py` can be overridden with a dotted key in `OVERRIDES`, e.g. `{'var.epochs': 20, 'data.max_samples': 20000, 'tagger.seeds': [42, 43, 44]}`.

**Experiment name.** `RUN_TAG` says what you are testing (short, e.g. `tokfix`, `posw3`, `lr2e-4`) and `RUN_DATE` is the day the experiment starts. `None` means today, which starts a **new** experiment. To **resume** after a disconnect on a later day, or to re-evaluate an old experiment, set `RUN_DATE` to that experiment's date; the cell lists the existing ones. One idea per experiment: change the tag when you change a setting.

Smoke and fake-data runs go to `MyDrive/GSoC_SuperRes/smoke_tests/` and a separate local cache, so they never mix with the real results.

In [ ]:
DATASET    = 'qg'     # 'qg' (CMS quark/gluon jets) or 'calo' (CaloChallenge DS2)
SMOKE      = False    # True: tiny subset + 1-2 epochs, to check the pipeline quickly
FAKE_DATA  = False    # synthetic data instead of the download (fastest check; implies SMOKE)
RUN_TUNING = False    # True: Optuna first (v1 studies are not reused; v2 has its own)
LEVELS     = ['pool2x2']   # start with 2x; None = all levels (the tokenizer always trains on all)
SEEDS      = [42, 43, 44]   # tagger seeds: 3 give an error bar from seed to seed; None = config value ([42])
VERSION    = 'v2'     # model version (see config.py)
RUN_TAG    = 'tokfix' # what this experiment tests, short; letters, digits, '-' and '.'
RUN_DATE   = None     # None = today -> new experiment. Resume / re-evaluate: its date, e.g. '2026-10-05'
TIMEZONE   = 'Asia/Kolkata'   # for "today" (Colab's clock is UTC)
OVERRIDES  = {
    # 'var.gen_splits': ['test'],      # physics check only; the taggers need ['train', 'val', 'test']
    # 'var.gen_max_per_split': 20000,  # caps the SR disk use (fp16: ~1 GB per 10k jets)
    # 'vqvae.hit_pos_weight': 3.0,     # raise if the tokenizer misses hits (val_hit_count_ratio well below 1)
}

import datetime, json, time
from zoneinfo import ZoneInfo
import pandas as pd
from superres import io_utils, pipeline
from superres.config import get_config
from superres.io_utils import RunDir

PROJECT = '/content/drive/MyDrive/GSoC_SuperRes'
SMOKE = SMOKE or FAKE_DATA
tag = '_fake' if FAKE_DATA else ('_smoke' if SMOKE else '')
RUN_ID = '{}-{}{}'.format(RUN_DATE or datetime.datetime.now(ZoneInfo(TIMEZONE)).strftime('%Y-%m-%d'),
                          RUN_TAG, tag.replace('_', '-'))
assert RUN_ID == io_utils.slug(RUN_ID), f"use only letters, digits, '-' and '.' in RUN_TAG: {RUN_TAG!r}"
paths = {
    'paths.drive_root': f"{PROJECT}/{'smoke_tests' if SMOKE else 'experiments'}",
    'paths.raw_root':   '/content/data/raw_fake' if FAKE_DATA else '/content/data/raw',
    'paths.cache_root': f'/content/data/cache{tag}',
}
cfg = get_config(DATASET, {**paths, 'version': VERSION, 'run_id': RUN_ID, **OVERRIDES}, smoke=SMOKE)

def stage(name, **kw):
    t0 = time.time()
    print(f'######## {DATASET}: {name} ########')
    out = pipeline.run_stage(cfg, name, levels=LEVELS, seeds=SEEDS, **kw)
    print(f'-- {name} finished in {(time.time() - t0) / 60:.1f} min')
    return out

def history(stage_name, variant, seed=cfg['seed'], n=5):
    """Last epochs of a run, plus its training-curve figure (saved on Drive next to the run)."""
    from IPython.display import Image, display
    run = RunDir(cfg, stage_name, variant, seed)
    fig = os.path.join(run.fig_dir, f'{run.name}__training_curves.png')
    if os.path.exists(fig):
        display(Image(fig, width=900))
    path = run.file('history.csv')
    return pd.read_csv(path).tail(n) if os.path.exists(path) else 'no history yet'

version_dir = os.path.join(cfg['paths']['drive_root'], DATASET, VERSION)
existing = sorted(os.listdir(version_dir)) if os.path.isdir(version_dir) else []
print('existing experiments in', version_dir, ':', existing or 'none')
print('this experiment:', RUN_ID, '(resuming)' if RUN_ID in existing else '(new)')

info = io_utils.save_run_info(
    cfg, notebook='superres_colab_v2.ipynb', branch=BRANCH, commit=git('rev-parse', 'HEAD'),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    smoke=SMOKE, fake_data=FAKE_DATA, levels=LEVELS, seeds=SEEDS, overrides=OVERRIDES)
print('experiment started:', info['started'], '| sessions so far:', len(info['sessions']))
print('dataset :', DATASET, '| smoke:', SMOKE, '| fake data:', FAKE_DATA)
print('levels  :', cfg['data']['levels'])
print(json.dumps(cfg['paths'], indent=1))
print('results :', io_utils.experiment_root(cfg))

## 3. Download

* `calo`: `dataset_2_1.hdf5` (train/val) and `dataset_2_2.hdf5` (test) from Zenodo record 6366271, about 1.4 GB each, md5-checked.
* `qg`: every file in the public CERNBox share. The share is listed over WebDAV and each file fetched with `wget`. If that fails, archive download of the whole share is tried. If both fail, paste direct file links into `OVERRIDES = {'data.extra_urls': [...]}` or copy the files into `/content/data/raw/qg/` yourself.

Downloads resume (`wget -c`), and files that are already complete are skipped.

In [ ]:
if FAKE_DATA:
    from tests.smoke_test import fake_calo, fake_qg
    raw = os.path.join(cfg['paths']['raw_root'], DATASET)
    if not (os.path.isdir(raw) and os.listdir(raw)):
        (fake_qg if DATASET == 'qg' else fake_calo)(raw)
    print(os.listdir(raw))
else:
    stage('download')

## 4. Prepare: energy-preserving down-sampling

Writes numpy memmaps to local disk: HR ground truth (zero-padded; qg 125 -> 128, calo 45 -> 48 layers) and each LR level made by **summing** blocks of cells, so every coarse cell holds exactly the energy of the fine cells it covers.

Check the printed label counts (qg) and `negative_fraction_clipped`. The step is skipped if the cache already exists; delete `/content/data/cache*/<dataset>` to rebuild it.

In [ ]:
meta = stage('prepare')
print(json.dumps({k: v for k, v in meta.items() if k != 'source_files'}, indent=1))
!du -sh {cfg['paths']['cache_root']}/{DATASET}

## 5. Multi-scale VQ-VAE (tokenizer)

Trained on HR images; the decoder gets the LR image of every level in turn. What to check in the printed rows and in the training-curve figure (saved to the run's `figures/` folder on Drive):

* `val_hit_count_ratio`: predicted / true number of hit cells. Should approach **1**. Well below 1 means hits are missed, so set `vqvae.hit_pos_weight` to about 3 and retrain; well above 1 means invented hits.
* `val_hit_precision`, `val_hit_recall`: fraction of predicted hits that are real / of real hits that are found.
* `val_channel_energy_err_median`: per-channel |E_pred − E_true| / E_true. Should be at most a few %.
* `codebook_usage`: fraction of codes in use. Very low means the codebook collapsed.

In [ ]:
if RUN_TUNING:
    stage('tune_vqvae')

In [ ]:
stage('train_vqvae')
history('vqvae', 'hr')

## 6. Conditional VAR transformer (one model per LR level)

Tuned once on the hardest level (depth with width = 64 x depth as in the paper, lr, dropout, weight decay, label smoothing; objective: validation token cross-entropy). The best parameters are reused for every level.

In [ ]:
if RUN_TUNING:
    stage('tune_var')

In [ ]:
stage('train_var')
for lvl in (LEVELS or cfg['data']['levels']):
    print(lvl); display(history('var', lvl))

## 7. Generate super-resolved images

The model output is stored exactly as generated. Nothing is corrected afterwards.

* The main output (`var.decode = 'greedy'`) takes the most likely token and is written for the splits in `var.gen_splits`. `sample` (tokens drawn from the predicted distribution) is written for the test split only, as a diagnostic.
* `var.gen_splits` chooses the splits. The physics check only needs `['test']`. The taggers need all three, and the training split is the large one (about 11 GB per level for all jets).
* Outputs that already exist are skipped. After retraining a model set `var.gen_overwrite` to `True`.

In [ ]:
stage('generate')
!du -sh {cfg['paths']['cache_root']}/{DATASET}

## 8. SR quality in physics terms

The main comparison is **HR truth vs LR vs the super-resolved output as generated**, on physics observables (jets: pT, mass, girth = mean ΔR, ΔR spread (`dR_std`), pT_D, tau21, hit count overall and per channel, radial profile; showers: E_dep / E_inc, shower depth and width, lateral centroid and width, energy fractions by depth, profiles).

For each quantity the table gives the mean for truth, LR and SR, and the distance of the SR distribution from the truth (**W1 / sigma_HR**, 0 = identical). Also reported: the **classifier two-sample test AUC** (0.5 = a CNN cannot tell SR from HR) and the **LR closure** (how well SR, summed back to coarse cells, agrees with the measurement).

Diagnostics are kept apart in `figures/diagnostics/` and as extra table columns:

* **Tokenizer only**: HR → tokens → HR with no transformer. This is the best the transformer could ever do with this tokenizer.
* **SR, other decoding**: greedy vs sampled tokens, if both were generated.
* **Uniform upsample**: coarse energy spread evenly, a no-learning reference.
* **SR + energy constraint**: SR rescaled so each coarse cell matches its measured energy.

**Event by event.** SR and HR are the same jets, so besides the distributions each observable is compared jet by jet (`paired.csv` in the `sreval` run, and a table in the report): **bias** = mean(SR − HR) / mean(HR), **resolution** = std(SR − HR) / mean(HR), and the correlation. Scatter plots of SR against HR (`__scatter__*`) are made for the track energy, total energy, pT, mass, girth, ΔR spread and hit count.

Histograms span all methods (0.5–99.5 % of each), so a shifted distribution appears where it is; the legend gives the fraction of events outside the range. Hit counts and pT_D depend on the cell size, so LR (on its coarse grid) is left out of those.

In [ ]:
from superres.evaluate import values_table

sr_tables = stage('eval_sr')
lvl = (LEVELS or list(cfg['data']['levels']))[-1]
values_table(sr_tables[lvl], lvl).round(4)   # mean of each quantity: truth | LR | SR | diagnostics

### Is the tokenizer or the transformer the limit?

Per-channel events with two extra rows: **Tokenizer only** (HR → tokens → HR, no transformer) and **SR, sampled tokens**.

* Tokenizer row close to HR, SR row not: the transformer is the limit.
* Tokenizer row already wrong: the tokenizer is the limit.

These figures are in `figures/diagnostics/` of the `sreval` run on Drive.

In [ ]:
import glob
from IPython.display import Image, display
for lvl in (LEVELS or list(cfg['data']['levels'])):
    d = os.path.join(RunDir(cfg, 'sreval', lvl).fig_dir, 'diagnostics')
    for f in sorted(glob.glob(os.path.join(d, '*__channels__event*.png')))[:2]:
        print(f); display(Image(f, width=700))

## 9. Downstream taggers

The architecture is tuned once on HR and the same one is trained separately on each input: **HR**, each **LR** grid, and each **SR** output as generated. Any difference in score then comes from the images, not from tuning effort. The question is whether SR closes the gap between LR and HR.

* `qg`: quark vs gluon classifier. Reports ROC AUC and background rejection 1/eps_B at signal efficiency 30/50/70%, with the uncertainty from resampling the test set (`boot_err`).
* `calo`: incident-energy regressor. Reports response bias and resolution sigma_eff/E, overall and in energy bins.

Every input is trained and tested on the **same events** (if SR was generated for fewer events, all inputs are cut to that number). Taggers that already finished are skipped, so the cell can be re-run after a disconnect; set `OVERRIDES['tagger.retrain'] = True` to redo them. Runs made before this rule existed are retrained automatically.

This stage needs SR for the train and validation splits too (`var.gen_splits`). To train on other inputs, pass them explicitly, e.g. `stage('train_taggers', kinds=['uniform:pool4x4', 'srproj:pool4x4'])`.

In [ ]:
if RUN_TUNING:
    stage('tune_tagger')

In [ ]:
stage('train_taggers')

### Tagger checks: training curves and the 2×3 table

**Training curves.** One figure per level with the taggers on HR, LR and SR overlaid (one colour per input, a line per seed): training and validation loss, accuracy and ROC AUC. The dot marks the epoch whose weights were kept (best validation AUC). Look for: validation loss rising while training loss falls (over-fitting), and whether the gap between inputs is larger than the spread between seeds.

**2×3 table.**

| evaluated on | A: one tagger trained on HR | B: a tagger trained on each input |
|---|---|---|
| HR | HR tagger on HR | HR tagger on HR |
| LR | HR tagger on LR stretched back to the HR grid (each coarse cell's energy spread evenly over its fine pixels: a blurred HR) | LR tagger on LR |
| SR | HR tagger on SR | SR tagger on SR |

* A(SR) ≈ A(HR) and A(LR) lower: SR gives back the fine detail the HR tagger relies on.
* A(SR) clearly below B(SR): the SR images carry the information but look different from HR (a domain shift).
* Column B alone can't separate these, because each model adapts to its own input.

In [ ]:
xeval = stage('tagger_xeval')
from IPython.display import Image, display
for lvl, table in xeval.items():
    print('==', lvl); display(table)
    for name in ('tagger_curves', 'tagger_2x3'):
        f = os.path.join(io_utils.summary_dir(cfg, 'figures'), f'{io_utils.prefix(cfg)}__{name}__{lvl}.png')
        if os.path.exists(f):
            display(Image(f, width=1000 if name == 'tagger_curves' else 600))

## 10. Summary tables and figures (saved to Drive under `<experiment folder>/summary/`)

In [ ]:
import glob
from IPython.display import Image, Markdown, display

stage('summarize')
sdir = io_utils.summary_dir(cfg)
display(Markdown(open(os.path.join(sdir, f'{io_utils.prefix(cfg)}__report.md')).read()))
for f in sorted(glob.glob(os.path.join(sdir, 'figures', '*.png'))):
    print(os.path.basename(f)); display(Image(f, width=560))

In [ ]:
# Per-level figures, all saved on Drive (png + pdf): histograms with ratio panels, profiles,
# example events and per-channel events.
for lvl in (LEVELS or list(cfg['data']['levels'])):
    fig_dir = RunDir(cfg, 'sreval', lvl).fig_dir
    print('==', lvl, '->', fig_dir)
    for f in sorted(glob.glob(os.path.join(fig_dir, '*.png'))):
        print(os.path.basename(f)); display(Image(f, width=420 if '__hist__' in f or '__profile__' in f else 700))

## 11. Housekeeping

The local cache is lost when the runtime is recycled; results on Drive are kept. To switch datasets in the same runtime, free the disk first.

In [ ]:
# !rm -rf /content/data/cache/{DATASET} /content/data/raw/{DATASET}
!df -h /content | tail -1
!du -sh /content/data/* 2>/dev/null